# **GPU Check + Mount Drive**


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

!pip install ultralytics -q

Mounted at /content/drive
GPU: Tesla T4
VRAM: 14.6 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.0 MB/s eta 0:00:00


# **Download ISIC 2018 Directly to Colab**

In [2]:
import os
os.makedirs('/content/data/images_raw', exist_ok=True)
os.makedirs('/content/data/labels_raw', exist_ok=True)

# Download Task 3 images (2.6GB) — direct from ISIC
!wget -q --show-progress -O /content/data/ISIC2018_Task3_Input.zip \
  "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task3_Training_Input.zip"

# Download Task 3 labels (36KB)
!wget -q --show-progress -O /content/data/ISIC2018_Task3_GT.zip \
  "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task3_Training_GroundTruth.zip"

print("Unzipping...")
!unzip -q /content/data/ISIC2018_Task3_Input.zip -d /content/data/images_raw/
!unzip -q /content/data/ISIC2018_Task3_GT.zip    -d /content/data/labels_raw/

print("Done!")
!ls /content/data/images_raw/ | head -5
!ls /content/data/labels_raw/

/content/data/ISIC2 100%[===================>]   2.58G  15.9MB/s    in 2m 41s  
/content/data/ISIC2 100%[===================>]  35.82K   166KB/s    in 0.2s    
Unzipping...
Done!
ISIC2018_Task3_Training_Input
ISIC2018_Task3_Training_GroundTruth


# **Prepare Dataset (Saliency BBoxes)**

In [3]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

IMAGES_DIR = Path("/content/data/images_raw/ISIC2018_Task3_Training_Input")
CSV_PATH   = Path("/content/data/labels_raw/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth.csv")
OUTPUT_DIR = Path("/content/data/yolo")

CLASS_NAMES   = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
CLASS_COUNTS  = {'MEL':1113,'NV':6705,'BCC':514,'AKIEC':327,'BKL':1099,'DF':115,'VASC':142}
total         = sum(CLASS_COUNTS.values())
CLASS_WEIGHTS = {i: round(total/(len(CLASS_COUNTS)*v),3) for i,(k,v) in enumerate(CLASS_COUNTS.items())}

def saliency_bbox(img):
    img_h, img_w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    sat = hsv[:,:,1]
    val = hsv[:,:,2]
    _, sat_mask = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    dark_mask   = (val < 80).astype(np.uint8) * 255
    combined    = cv2.bitwise_or(sat_mask, dark_mask)
    kernel      = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15,15))
    cleaned     = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel)
    cleaned     = cv2.morphologyEx(cleaned,  cv2.MORPH_OPEN,  kernel)
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.5, 0.5, 0.7, 0.7
    largest  = max(contours, key=cv2.contourArea)
    area     = cv2.contourArea(largest)
    img_area = img_w * img_h
    if area < 0.01*img_area or area > 0.95*img_area:
        return 0.5, 0.5, 0.7, 0.7
    x, y, w, h = cv2.boundingRect(largest)
    pad_x = int(w*0.08); pad_y = int(h*0.08)
    x = max(0, x-pad_x); y = max(0, y-pad_y)
    w = min(img_w-x, w+2*pad_x); h = min(img_h-y, h+2*pad_y)
    return (x+w/2)/img_w, (y+h/2)/img_h, w/img_w, h/img_h

df = pd.read_csv(CSV_PATH)
df['class_id'] = df[CLASS_NAMES].values.argmax(axis=1)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class_id'])

for split in ['train','val']:
    (OUTPUT_DIR/split/'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR/split/'labels').mkdir(parents=True, exist_ok=True)

for split, split_df in [('train',train_df),('val',val_df)]:
    print(f"Processing {split} ({len(split_df)} images)...")
    for i, (_, row) in enumerate(split_df.iterrows()):
        img_id   = row['image']
        class_id = row['class_id']
        img_src  = IMAGES_DIR / f"{img_id}.jpg"
        if not img_src.exists(): continue
        img = cv2.imread(str(img_src))
        if img is None: continue
        cx, cy, w, h = saliency_bbox(img)
        img_resized  = cv2.resize(img, (640, 640))
        cv2.imwrite(str(OUTPUT_DIR/split/'images'/f"{img_id}.jpg"), img_resized,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        with open(OUTPUT_DIR/split/'labels'/f"{img_id}.txt", 'w') as f:
            f.write(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
        if i % 1000 == 0:
            print(f"  {i}/{len(split_df)}")

print("✅ Dataset ready!")

Processing train (8012 images)...
  0/8012
  1000/8012
  2000/8012
  3000/8012
  4000/8012
  5000/8012
  6000/8012
  7000/8012
  8000/8012
Processing val (2003 images)...
  0/2003
  1000/2003
  2000/2003
✅ Dataset ready!


# **Write skin.yaml**

In [4]:
yaml_content = """path: /content/data/yolo
train: train/images
val: val/images

nc: 7
names:
  0: MEL
  1: NV
  2: BCC
  3: AKIEC
  4: BKL
  5: DF
  6: VASC
"""
with open('/content/data/skin.yaml', 'w') as f:
    f.write(yaml_content)
print("✅ skin.yaml written")

✅ skin.yaml written


# **Train with Save-Every-Epoch to Drive**

**V2 training**

In [ ]:
from ultralytics import YOLO
import shutil, os, torch

CLASS_NAMES   = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
CLASS_WEIGHTS = [1.285, 0.213, 2.783, 4.375, 1.302, 12.441, 10.075]
DRIVE_WEIGHTS = '/content/drive/MyDrive/skin_detection/weights2'
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB\n")

model = YOLO('yolo11s.pt')

def save_to_drive(trainer):
    epoch   = trainer.epoch + 1
    best    = '/content/runs/skin_v2/weights/best.pt'   # ← fixed
    last    = '/content/runs/skin_v2/weights/last.pt'   # ← fixed
    metrics = trainer.metrics

    if os.path.exists(best):
        shutil.copy(best, f'{DRIVE_WEIGHTS}/best.pt')
    if os.path.exists(last):
        shutil.copy(last, f'{DRIVE_WEIGHTS}/last.pt')

    if epoch % 5 == 0 and os.path.exists(best):
        shutil.copy(best, f'{DRIVE_WEIGHTS}/best_epoch{epoch:03d}.pt')
        print(f"  📦 Checkpoint saved: best_epoch{epoch:03d}.pt")

    map50 = metrics.get('metrics/mAP50(B)', 0)
    print(f"  💾 Epoch {epoch:>3} → mAP50: {map50:.4f} — saved to Drive")

model.add_callback('on_fit_epoch_end', save_to_drive)

results = model.train(
    data          = '/content/data/skin.yaml',
    imgsz         = 640,
    epochs        = 80,
    patience      = 15,
    batch         = 32,
    workers       = 2,
    device        = 'cuda',
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,
    weight_decay  = 0.0005,
    cos_lr        = True,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 15.0,
    fliplr        = 0.5,
    flipud        = 0.5,
    mosaic        = 0.5,
    mixup         = 0.05,
    cls           = 0.5,
    project       = '/content/runs',
    name          = 'skin_v2',
    exist_ok      = True,
    verbose       = True,
    plots         = True,
)

print("\n✅ Training complete!")
print(f"Best weights on Drive: {DRIVE_WEIGHTS}/best.pt")

In [6]:
metrics = model.val(
    data    = '/content/data/skin.yaml',
    augment = True,
    workers = 2,
)

CLASS_NAMES = ['MEL','NV','BCC','AKIEC','BKL','DF','VASC']

print("="*50)
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
print("\nPer-class AP@0.5:")
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    bar = "█" * int(ap * 30)
    print(f"  {name:<8} {ap:.4f}  {bar}")

# Save final results to Drive
import shutil
shutil.copy('/content/runs/skin_v1/results.png',
            '/content/drive/MyDrive/skin_detection/weights/results.png')
print("\n✅ Results plot saved to Drive")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,415,509 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2304.0±727.5 MB/s, size: 85.3 KB)
val: Scanning /content/data/yolo/val/labels.cache... 2003 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2003/2003 763.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 126/126 2.2it/s 56.8s
                   all       2003       2003      0.494      0.593       0.55       0.48
                   MEL        223        223      0.424      0.583      0.522       0.47
                    NV       1341       1341      0.788      0.957      0.949      0.884
                   BCC        103        103      0.418      0.592      0.531       0.42
                 AKIEC         65         65      0.285      0.508      0.293      0.245
                   BKL        220     